# LOOP module — Trend Radar $100k (S&P 500 + Nasdaq-100, no pharma)

Loads LOOP run logs from Supabase and evaluates the paper book against **SPY** (S&P 500) and **QQQ** (Nasdaq-100) benchmarks.

Tradable universe policy (see `pipeline/ingest/us_index_universe.py`):
- **Include:** S&P 500 ∪ Nasdaq-100
- **Exclude:** pharmaceutical / biotechnology names


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.research.loop_log_report import (
    benchmark_index_curve,
    closed_trades,
    equity_curve_from_prices,
    flatten_run_logs,
    get_client,
    load_loop_tables,
    load_universe_map,
    portfolio_snapshot,
    without_pharma,
)

pd.set_option("display.max_colwidth", 120)


In [ ]:
tables = load_loop_tables()
sb = get_client()
uni = load_universe_map(sb)

orders_all = tables["orders"]
book_all = tables["book"]
orders, book = without_pharma(orders_all, book_all, uni)

runs_df = pd.DataFrame(tables["runs"])
logs_df = flatten_run_logs(tables["runs"])

print(f"Runs: {len(runs_df)} | Orders (all): {len(orders_all)} | Orders (ex-pharma): {len(orders)}")
print(f"Open book (all): {len(book_all)} | Open book (ex-pharma): {len(book)}")
if book_all:
    dropped = sorted({p['symbol'] for p in book_all} - {p['symbol'] for p in book})
    print('Pharma names removed from analysis:', dropped)


In [ ]:
snapshot_all = portfolio_snapshot(orders_all, book_all)
snapshot = portfolio_snapshot(orders, book)
pd.DataFrame([
    {'scope': 'All LOOP history', **snapshot_all},
    {'scope': 'Ex-pharma (target universe)', **snapshot},
]).set_index('scope')


In [ ]:
closed = closed_trades(orders)
closed


In [ ]:
curve = equity_curve_from_prices(sb, orders)
if not curve.empty:
    start, end = curve['date'].min(), curve['date'].max()
    bench = benchmark_index_curve(sb, start, end)
else:
    bench = pd.DataFrame()
curve.tail(), bench.tail() if not bench.empty else bench


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

if not curve.empty:
    axes[0, 0].plot(curve['date'], curve['return_pct'], marker='o', label='LOOP (ex-pharma)')
    if not bench.empty:
        for col in [c for c in bench.columns if c != 'date']:
            axes[0, 0].plot(bench['date'], bench[col] - 100, label=col)
    axes[0, 0].axhline(0, color='gray', ls='--', lw=1)
    axes[0, 0].set_title('Return % vs SPY / QQQ')
    axes[0, 0].legend()
    axes[0, 0].tick_params(axis='x', rotation=45)

if not runs_df.empty:
    axes[0, 1].plot(runs_df['asof_date'], runs_df['deployed_pct'] * 100, marker='o')
    axes[0, 1].set_title('Deployed % (all runs)')
    axes[0, 1].tick_params(axis='x', rotation=45)

if not closed.empty:
    axes[1, 0].bar(closed['ticker'], closed['pnl_usd'], color=['#2ecc71' if x > 0 else '#e74c3c' for x in closed['pnl_usd']])
    axes[1, 0].set_title('Realized PnL (ex-pharma)')
    axes[1, 0].axhline(0, color='black', lw=0.8)

if not logs_df.empty:
    pharma_skips = logs_df['message'].str.contains('PHARMA_EXCLUDED', na=False).sum()
    axes[1, 1].bar(['SKIP', 'ORDER', 'PHARMA_EXCLUDED'], [
        (logs_df['level'] == 'SKIP').sum(),
        (logs_df['level'] == 'ORDER').sum(),
        pharma_skips,
    ])
    axes[1, 1].set_title('Log activity')

plt.tight_layout()
out = ROOT / 'pipeline/research/artifacts/loop_analysis_charts.png'
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=144)
plt.show()


In [ ]:
ret = snapshot['total_return_pct']
tone = 'OK' if ret >= 0 else ('MIXED' if ret > -5 else 'STRESSED')
print(f'Health (ex-pharma LOOP vs $100k): {tone} · {ret:+.2f}% MTM')
print(f"Realized ${snapshot['realized_pnl_usd']:+,.0f} · Unrealized ${snapshot['unrealized_pnl_usd']:+,.0f}")
if not bench.empty and not curve.empty:
    last = bench.iloc[-1]
    print(f"Same window: SPY {last.get('SPY', float('nan')) - 100:+.2f}% · QQQ {last.get('QQQ', float('nan')) - 100:+.2f}%")
